In [1]:
import pandas as pd
import MaxNLP
from pathlib import Path
from tqdm.auto import tqdm
import json
tqdm.pandas(desc="Applying")
%load_ext autoreload
%autoreload 2
    
from datetime import date
date_str = date.today().isoformat()

In [3]:
# Load the translated text sample (25 per country --> 250 samples)
df_t=pd.read_json("2026-05-02 new translated.json")
#df=pd.read_json("clean_df.json")

#df.columns[-20:]
#df["full_response"] = df.apply(MaxNLP.make_one_text, axis=1)

In [19]:
# Read the codebooks as Markdown

codebook_causes = Path("codebooks/2026-05-13 causes codebook.md").read_text(encoding="utf-8")
codebook_responses = Path("codebooks/2026-01-30 responses codebook.md").read_text(encoding="utf-8")
codebook_consequences = Path("codebooks/2026-04-08b consequences codebook.md").read_text(encoding="utf-8")
codebook_futures = Path("codebooks/2026-02-15 lessons codebook.md").read_text(encoding="utf-8")

# Convert Markdown to Json
codebook_causes_json=MaxNLP.markdown_to_json(codebook_causes)
codebook_responses_json = MaxNLP.markdown_to_json(codebook_responses)
codebook_consequences_json=MaxNLP.markdown_to_json(codebook_consequences)
codebook_futures_json=MaxNLP.markdown_to_json(codebook_futures)


# Get the code list for creating the json scheme.
codes_causes= [i["label"] for i in codebook_causes_json]
codes_responses = [i["label"] for i in codebook_responses_json]
codes_consequences = [i["label"] for i in codebook_consequences_json]
codes_futures = [i["label"] for i in codebook_futures_json]


## Important: Context window is set to 2000 = roughly 2000 words; larger codebooks require to change this.

print(len(codebook_causes.split()))

for i in codebook_causes_json:
    print(i["label"])
    print(len(i['description'].split()),len(i['inclusion_criteria']),len(i['exclusion_criteria']),len(i['included_examples']))

2800
EU political actors
39 7 6 7
National political actors
39 8 6 4
Industry and market actors
26 2 5 4
Media actors
26 2 4 2
Russia's war on Ukraine
13 2 4 4
Too-fast energy transition
14 3 2 2
Dependence on fossil fuels
22 4 4 8
Concentrated energy import dependency
26 3 2 5
Uncontrollable natural disasters
26 4 3 3
External shocks and technical failures
35 6 6 6
Abstract societal and ideological explanations
45 4 5 5
EU sanctions against Russia
25 4 4 6
nuclear phase-out / reduced national production
9 2 3 4
Insufficient price regulation
13 4 3 11


In [21]:
# Create the System prompt --> requires to create a new client when changing the codebook.


PROMPTBOOK_INSTRUCTIONS = """
You are helpful qualitative research assistant coding short citizen perceptions of the European energy crisis with a defined codebook.
For each label, the codebook includes a definition with inclusion/exclusion criteria and examples that are coded with this label. 
You carefully read the full citizen perception and apply the codebook: What does the citizen say?
You only respond in correct json format. Task: For EACH label, output a confidence score (0.0 = impossible, 1.0 = certain). 
Multi-label allowed - some codes specify general ones. 
Return JSON only, matching the schema exactly (no extra keys). 
Do not add any labels that are not in the schema.
This is your codebook for the question "What caused the EU energy crisis?":\n
""".strip()

In [23]:
# Load the Nebula Client
from dotenv import load_dotenv
import openAI_key

client=MaxNLP.create_client(client="Nebula", openAI_key=openAI_key)
available_models = MaxNLP.get_nebula_models(client)

print(f"Models: {available_models}\n\n")

run_x=1

Nebula client loaded
Models: ['deepseek-r1:8b', 'deepseek-r1:1.5b', 'SURF.default-text-large', 'FAST.gemma3:12b', 'FAST.gpt-oss:120b', 'FAST.gpt-oss:20b', 'go-assist', 'research-railway-guide', 'SURF.Qwen 2.5 Coder 32B Instruct AWQ', 'SURF.Qwen 2.5 VL 32B Instruct AWQ', 'vu-rdm-support-chatbot---test', 'translategemma:12b', 'SURF.Mistral-Small-3.2-24B-Instruct-2506', 'SURF.Qwen3.5 122B A10B NVFP4', 'SURF.gpt-oss-120b', 'llama3.1:8b']




Code for manually combining files


ann_df = (
    pd.DataFrame(
        x
        for f in Path("annotations_tmp").glob(f"2026-04-23 run_1_causes_batch_*.jsonl")
        for x in jsonlines.open(f)
    )
    .pipe(lambda d:
        pd.json_normalize(d["annotation"])
        .set_index(d["orig_index"])
        .loc[lambda df: ~df.index.duplicated(keep="last")]
    )
)
#ann_df.reset_index().to_json(f"2026-04-23 causes_run_1.json", orient="records")

In [25]:
# Run the LLM coding in a for-loop

codebook_name="causes"
code_list=codes_causes
codebook=json.dumps(codebook_causes_json, indent=2, ensure_ascii=False)
text_column='QID8'

import pandas as pd
import jsonlines
from pathlib import Path

for run_x in range(5):

    # Split the set in batches (to rerun not all of them when there is an error)
    tmp = df_t.reset_index(names="orig_index") ### df_t is the translated sample; df is the full sample.
    tmp["batch_row"] = range(len(tmp))
    #run_x=f'{run_x}b'
    
    # Send the API request to the LLM; format everything with my module.
    # Write each result as new line to the temp  batch files 
    res = tmp.progress_apply(
        lambda r: MaxNLP.code_text(
            r[text_column], r.orig_index, r.batch_row,
            client=client,
            model="FAST.gpt-oss:120b",
            code_list=code_list,
            PROMPTBOOK_INSTRUCTIONS=PROMPTBOOK_INSTRUCTIONS+codebook,
            temperature=0.0,
            force_json_object=False,
            run_x=run_x,
            codebook_name=codebook_name
        ),
        axis=1,
    )

    # Combine the batch files to one json file
    ann_df = pd.DataFrame(x for f in Path("annotations_tmp").glob(f"{date_str} run_{run_x}_{codebook_name}_batch_*.jsonl") for x in jsonlines.open(f)).pipe(
        lambda d: pd.json_normalize(d["annotation"])
        .set_index(d["orig_index"]))
    
    # Writes the full run to one json file
    ann_df.reset_index().to_json(f"{date_str} {codebook_name}_run_{run_x}.json", orient="records")


Applying:   0%|          | 0/250 [00:00<?, ?it/s]

Applying:   0%|          | 0/250 [00:00<?, ?it/s]

Applying:   0%|          | 0/250 [00:00<?, ?it/s]

Applying:   0%|          | 0/250 [00:00<?, ?it/s]

Applying:   0%|          | 0/250 [00:00<?, ?it/s]

# Evaluation

In [ ]:
# Check Variation --> indicated stability of LLM runs.

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
filelist=[pd.read_json(f"2026-04-08 {codebook_name}_run_{i}.json").set_index("orig_index").ge(0.5).astype(int) for i in range(4)]+(
[pd.read_json(f"2026-04-10 {codebook_name}_run_{i}.json").set_index("orig_index").ge(0.5).astype(int) for i in range(5)])


x = pd.concat(filelist,
    keys=range(len(filelist)), names=["run"])



alphas = MaxNLP.kripp_alpha_all_variables(x, decimals=2)

alphas


#x_mode = x.groupby("orig_index").agg(lambda s: s.mode().tolist())
x_mode = x.groupby("orig_index").agg(lambda s: s.mode().iat[0])
x_mode.sum().sort_values().sort_values().plot.barh()


In [ ]:
#Load Manual coding as gold standard
Annotation=pd.read_excel("MANUAL CODING/2026-04-07 consequences Manual Annotation filled.xlsx", sheet_name="Sheet1",header=0).T
#Annotation=pd.read_excel("MANUAL CODING/2026-03-16 causes Manual Annotation filled.xlsx", sheet_name="Sheet1",header=0).T
Annotation.columns=Annotation.iloc[0,:]
Text=Annotation.iloc[1:,-4]
Annotation=Annotation.iloc[1:,:-4] # remove the text, translation, and notex

n_samples=Annotation.shape[0]

import matplotlib.pyplot as plt

s1 = Annotation.sum().sort_values()
s1.plot.barh(title=f"Causes of the Crisis (manual coding, n={n_samples})" , figsize=(3,3))
plt.tight_layout()
plt.show()
pd.set_option('display.max_colwidth', None)  # or a large number like 1000



In [ ]:
merged = pd.concat(
    [x_mode.set_axis(x_mode.index.astype(int)).add_prefix("ai_"),
     Annotation.set_axis(Annotation.index.astype(int)).add_prefix("manual_")],
    axis=1,
    join="inner"
)

merged.columns

In [ ]:
per_topic = MaxNLP.eval_ai_vs_manual(merged, question_list=x_mode.columns)
per_topic

In [ ]:
text=df_t["full_response"].rename("text")
merged_t = merged.join(text)

out_path = f"{date_str} causes_False Positives.txt"

with open(out_path, "w", encoding="utf-8") as f:
    for col in x_mode.columns:
        ai_col = f"ai_{col}"
        manual_col = f"manual_{col}"

        mask = merged_t[ai_col].astype(int) != (merged_t[manual_col].astype(float) >= 0.5).astype(int)
        false_ones = merged_t.loc[mask, "text"]
        f.write(f"{col}:\n")
        for n, text in enumerate(false_ones, 1):
            f.write(f"  {n}. {str(text).strip()}\n")
        f.write("\n\n")

print(f"Written to {out_path}")